# Week 2 — live-coding notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week02/week2_demos.ipynb)

At the first break, run any **Setup** cell and then the **Imports** cell. If there is no Setup cell, begin with Imports. After that, use the slide cue to jump to the named demo; you do not need to rerun the whole notebook at every break. To check the whole notebook from a clean start, use **Runtime → Run all**.

Each demo follows the same rhythm: predict what the output should show, run the cell, and follow the task immediately underneath it. When an editable research-record entry appears, its completed comparison sits below it in a collapsed box—write your version before opening that box. This notebook is generated from the same source code the lecturer runs live.

## Jump to a demonstration

Use these links during the lecture breaks; you do not need to scroll through or rerun the entire notebook.

### Deck A

- [Demo 1 — The missing-shift trap](#demo-1-the-missing-shift-trap)
- [Demo 2 — Build the five feature families on a simulated price path](#demo-2-build-the-five-feature-families-on-a-simulated-price-path)
- [Real-data companion — apply the same close-to-close clock to market returns](#real-data-companion-apply-the-same-close-to-close-clock-to-market-returns)
- [Demo 3 — Time-series scaling and cross-sectional comparison](#demo-3-time-series-scaling-and-cross-sectional-comparison)
- [Demo 4 — Define and align a forward-return target](#demo-4-define-and-align-a-forward-return-target)
- [Demo 5 — Look-ahead self-check on the full feature matrix](#demo-5-look-ahead-self-check-on-the-full-feature-matrix)

### Deck B

- [Demo 6 — Manufacture a leak, then measure it](#demo-6-manufacture-a-leak-then-measure-it)
- [Demo 7 — Full-sample normalization is an invisible leak](#demo-7-full-sample-normalization-is-an-invisible-leak)
- [Demo 8 — Point-in-time alignment with an as-of join](#demo-8-point-in-time-alignment-with-an-as-of-join)
- [Demo 9 — The look-ahead self-check](#demo-9-the-look-ahead-self-check)
- [Demo 10 — A spectacular backtest built entirely from a leak](#demo-10-a-spectacular-backtest-built-entirely-from-a-leak)

## Your Week 2 practice research record

Complete each entry in the editable Markdown cell immediately below its demo. **Do not write in this overview.** The local entries together form your Week 2 practice record. They remain in this notebook and are not submitted separately.

This week, every entry should make three things visible: what information was used, when it was available, and what later outcome or comparison is allowed to judge it.

### The six lecture breaks

1. **Features — Demos 1–2:** record the missing-shift result and complete the two six-field feature definitions.
2. **Comparison sets — Demo 3:** record one asset under its own-history, peer, and known-beta comparisons, plus the frozen scaler cutoff.
3. **The modeling row — Demos 4–5:** audit one complete timestamp ledger, its first and last valid decisions, and the future-invariance result.
4. **A manufactured leak — Demo 6:** compare the leaked and repaired forward R² values in a known-truth experiment.
5. **Fitted preprocessing — Demo 7:** record which population fitted each ruler and why the first test value changes.
6. **Point-in-time proof — Demos 8–9:** complete the seven-field boundary record after the grouped as-of join, real ALFRED GDP vintage comparison, dated-universe check, and future-invariance test.

The real-market companion after Demo 2 is optional and has its own clock-specific entry. For every result, include one limitation and only the narrow conclusion the evidence supports.

In [ ]:
# Setup: install packages not preinstalled in Colab.
%pip install -q finmlsim

### Imports

> **Run this once before any demo.** It loads the packages and helper functions used below; there is no result to interpret in this cell.

In [ ]:
from io import BytesIO, StringIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import RobustScaler, StandardScaler

from finmlsim.simulate import garch


def course_csv(relative_path):
    """Read bundled Fama-French data locally, or its public source in Colab."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local = root / relative_path
        if local.exists():
            return pd.read_csv(local), str(local)
    archives = {
        "datasets/famafrench/ff_factors_daily.csv": "F-F_Research_Data_Factors_daily_CSV.zip",
        "datasets/famafrench/ff_12industry_daily.csv": "12_Industry_Portfolios_daily_CSV.zip",
    }
    archive = archives[relative_path]
    url = f"https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/{archive}"
    with urlopen(url) as response, ZipFile(BytesIO(response.read())) as zipped:
        lines = zipped.read(zipped.namelist()[0]).decode("utf-8").splitlines()
    header_row = next(i for i, line in enumerate(lines) if line.startswith(","))
    rows = []
    for line in lines[header_row + 1:]:
        first = line.split(",", 1)[0].strip()
        if len(first) == 8 and first.isdigit():
            rows.append(line)
        elif rows:
            break
    frame = pd.read_csv(StringIO("\n".join([lines[header_row], *rows])))
    return frame.rename(columns={frame.columns[0]: "date"}), url


def bundled_csv(relative_path, embedded_csv):
    """Read a small lecture extract locally, with an offline notebook fallback."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local = root / relative_path
        if local.exists():
            return pd.read_csv(local), str(local)
    return pd.read_csv(StringIO(embedded_csv)), "embedded lecture extract"


def build_week2_features():
    """Construct the shared simulated return series and its feature matrix."""
    n_rows = 2000
    returns = pd.Series(
        garch(n=n_rows, omega=1e-6, alpha=0.07, beta=0.90, seed=1),
        name="close_to_close_return",
    )
    price = pd.Series(100 * np.exp(np.cumsum(returns)), name="close")

    features = pd.DataFrame(index=returns.index)
    # 1. Lagged returns
    for lag in [1, 2, 3, 5]:
        features[f"r_lag{lag}_t"] = returns.shift(lag)
    # 2. Momentum, built only from completed prior returns
    for window in [5, 21, 63]:
        features[f"mom{window}_t"] = returns.rolling(window).mean().shift(1)
    # 3. Volatility
    for window in [21, 63]:
        features[f"vol{window}_t"] = returns.rolling(window).std().shift(1)
    # 4. A rolling z-score whose reference history also stops before t
    mom21_known = returns.shift(1).rolling(21).mean()
    z_window = 252
    features["zmom21_t"] = (
        (mom21_known - mom21_known.rolling(z_window).mean())
        / mom21_known.rolling(z_window).std()
    )
    # 5. Calendar information is known in advance and needs no shift
    features["dow_t"] = np.arange(n_rows) % 5

    definitions = pd.DataFrame(
        [
            {
                "feature": "mom21_t",
                "source": "daily close-to-close log returns",
                "availability": "completed through t-1",
                "transformation": "21-day trailing mean",
                "memory": "equal weight over 21 sessions",
                "unit": "log return per day",
                "hypothesis": "short trend persists",
            },
            {
                "feature": "vol21_t",
                "source": "daily close-to-close log returns",
                "availability": "completed through t-1",
                "transformation": "21-day trailing sample standard deviation",
                "memory": "recent return magnitude and dispersion",
                "unit": "daily volatility",
                "hypothesis": "risk state persists",
            },
        ]
    )
    return returns, price, features, definitions


def build_week2_row_ledger():
    """Attach a feasible next-open-to-close target to the shared feature matrix."""
    returns, price, features, definitions = build_week2_features()
    sessions = pd.Series(
        pd.bdate_range("2015-01-02", periods=len(returns)),
        index=returns.index,
        name="session",
    )
    next_session = sessions.shift(-1)
    previous_session = sessions.shift(1)

    # This is a separate simulated next-open-to-close return. It matches the
    # target interval written into the ledger; it is not a close-to-close return
    # relabeled as though it began at the next open.
    intraday_return = pd.Series(
        garch(n=len(returns), omega=1e-6, alpha=0.07, beta=0.90, seed=2),
        index=returns.index,
        name="open_to_close_return",
    )
    target = intraday_return.shift(-1).rename("target_return")
    target_sign = (target > 0).where(target.notna()).rename("target_sign")

    ledger = features.copy()
    ledger.insert(0, "decision_time", sessions + pd.Timedelta(hours=16, minutes=5))
    ledger.insert(1, "latest_source_time", previous_session + pd.Timedelta(hours=16))
    ledger["earliest_entry"] = next_session + pd.Timedelta(hours=9, minutes=30)
    ledger["target_start"] = ledger["earliest_entry"]
    ledger["target_end"] = next_session + pd.Timedelta(hours=16)
    ledger["target_return"] = target
    ledger["target_sign"] = target_sign
    snapshot_time = sessions.iloc[-1] + pd.Timedelta(hours=16, minutes=5)
    ledger["label_matured"] = (
        ledger["target_end"].notna() & (ledger["target_end"] <= snapshot_time)
    )

    timeline = ledger.dropna(
        subset=["mom21_t", "vol21_t", "latest_source_time", "target_start", "target_end"]
    )
    model_columns = list(features.columns)
    aligned = ledger.dropna(subset=[*model_columns, "target_return"]).copy()
    audit_columns = [
        "decision_time", "latest_source_time", "earliest_entry",
        "target_start", "target_end", "target_return", "label_matured",
    ]
    return returns, features, ledger, timeline, aligned, audit_columns

<a id="demo-1-the-missing-shift-trap"></a>

### Demo 1 — The missing-shift trap · Deck A · run at the break after S1

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
# i.i.d. noise has zero properly shifted correlation; an unshifted rolling-mean feature
# manufactures spurious correlation out of thin air.
T = 5000
demo1_rng = np.random.default_rng(42)
noise = demo1_rng.standard_normal(T)
s = pd.Series(noise)

corrs = {}
for shift in range(0, 5):
    feat = s.rolling(5).mean().shift(shift)
    corrs[shift] = feat.corr(s)
print("Correlation of rolling(5).mean().shift(k) with same-day return on i.i.d. noise:")
for k, v in corrs.items():
    flag = "<-- LEAK (includes today)" if k == 0 else ""
    print(f"  shift={k}: corr = {v:+.3f}   {flag}")
# Expected: shift=0 gives corr ~ +0.45 on i.i.d. noise -- pure leak. Every shift >= 1
#           is ~0, which is the truth.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| CLAIM | EVIDENCE | BOUNDARY | LIMITATION |
|:--|:--|:--|:--|
| `[narrow conclusion]` | `[specific output, value, or comparison]` | `[sample, clock, fit cutoff, and what stayed fixed]` | `[what this exercise cannot establish]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed boundary entry**

| CLAIM | EVIDENCE | BOUNDARY | LIMITATION |
|:--|:--|:--|:--|
| The unshifted rolling mean contains the outcome and manufactures association; the shifted feature uses only prior returns under this daily clock. | Same-day correlation is **+0.451** unshifted and **+0.008** after one shift; $1/\sqrt{5}=0.447$ predicts the mechanical leak. | i.i.d. simulation, five-day window; permitted inputs end at *t*−1. | One simulation and window; correct timing does not prove the feature predicts anything. |

The honest conclusion is about construction, not market skill.

</details>

<a id="demo-2-build-the-five-feature-families-on-a-simulated-price-path"></a>

### Demo 2 — Build the five feature families on a simulated price path · Deck A · run at the break after S1

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
ret, price, feat, definitions = build_week2_features()

print(feat.tail())
print(f"\nFeature matrix shape after dropna: {feat.dropna().shape}")

print("\nTwo example feature definitions:")
print(definitions.to_string(index=False))

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

Complete this once for `mom21_t` and once for `vol21_t`.

| FIELD | YOUR ENTRY |
|:--|:--|
| source | `[raw values used]` |
| availability | `[latest admissible timestamp]` |
| transformation | `[formula]` |
| memory | `[what history it keeps]` |
| unit | `[measurement unit]` |
| hypothesis | `[why it might help]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed six-field feature definitions**

| FIELD | `mom21_t` | `vol21_t` |
|:--|:--|:--|
| source | daily close-to-close returns | daily close-to-close returns |
| availability | completed through *t*−1 | completed through *t*−1 |
| transformation | mean of the prior 21 returns | sample standard deviation of the prior 21 returns |
| memory | recent average direction | recent magnitude and dispersion |
| unit | return per day | return per square-root day |
| hypothesis | recent direction may persist | recent risk may persist |

Both definitions use only information allowed under this daily row convention. Neither claims
that the remembered quantity will improve a later forecast.

</details>

<a id="real-data-companion-apply-the-same-close-to-close-clock-to-market-returns"></a>

### Real-data companion — apply the same close-to-close clock to market returns · Deck A · optional after S1

> **Optional real-data clock check:** Deck A, after recording segment S1. Run the cell to apply the same prior-close feature cutoff and close-to-close target interval to dated US market returns. Record the clock and one printed correlation, but do not interpret the full-sample correlation as a backtest.

In [ ]:
ff_raw, ff_source = course_csv("datasets/famafrench/ff_factors_daily.csv")
ff_raw["date"] = pd.to_datetime(ff_raw["date"].astype(str), format="%Y%m%d")
ff_real = ff_raw.set_index("date").sort_index()
market_return = (ff_real["Mkt-RF"] + ff_real["RF"]) / 100.0

# Decision: just before the close on t, using data completed through t-1. A
# market-on-close entry at t can then earn the close-t to close-(t+1) return.
real_features = pd.DataFrame(index=market_return.index)
real_features["r_lag1_t"] = market_return.shift(1)
real_features["mom5_t"] = market_return.rolling(5).mean().shift(1)
real_features["mom21_t"] = market_return.rolling(21).mean().shift(1)
real_features["vol21_t"] = market_return.rolling(21).std().shift(1)
real_target = market_return.shift(-1).rename("next_return")
real_aligned = pd.concat([real_features, real_target], axis=1).dropna()

print("Source: Kenneth R. French Data Library, bundled daily factors")
print(f"File: {ff_source}")
print(f"Sample: {real_aligned.index.min().date()} through {real_aligned.index.max().date()}")
print("Clock: decide just before close t using data through close t-1; enter at close t")
print("Units: decimal daily returns; target is close t to close t+1")
print(real_aligned.tail(3).round(5).to_string())
print("\nFull-sample correlations shown only as a mechanical check, not a backtest:")
for column in real_features.columns:
    print(f"  corr({column:8s}, next_return) = {real_aligned[column].corr(real_aligned['next_return']):+.3f}")
print("The feature definition survives contact with real dates; predictive strength is a separate question.")

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| decision and entry | `[decision time, information cutoff, and feasible entry]` |
| feature cutoff | `[latest return used by each feature]` |
| target interval | `[start, end, and stored column]` |
| sample | `[source and first/last aligned dates]` |
| mechanical check | `[one or more printed correlations]` |
| limitation | `[what these correlations cannot establish]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed real-data clock entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| decision and entry | decide just before close *t* using data completed through close *t*−1; enter at close *t* |
| feature cutoff | `r_lag1_t`, `mom5_t`, `mom21_t`, and `vol21_t` all stop at *t*−1 |
| target interval | close *t* to close *t*+1, stored as `next_return` |
| sample | Kenneth R. French daily US market data, 1926-07-28 through 2026-04-29 |
| mechanical check | full-sample correlations with `next_return`: `r_lag1_t` **−0.025**, `mom5_t` **−0.014**, `mom21_t` **+0.006**, `vol21_t` **+0.009** |
| limitation | these full-sample correlations check alignment only; they are not a forward backtest or evidence of a tradeable signal |

The simulated and real examples now use the same clock. Whether any feature
predicts later returns remains a separate question.

</details>

<a id="demo-3-time-series-scaling-and-cross-sectional-comparison"></a>

### Demo 3 — Time-series scaling and cross-sectional comparison · Deck A · run at the break after S2

> **Break cue:** Deck A, after recording segment S2. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
# A simulated panel where assets have different exposures to one market factor.
# Time-series scaling compares each asset with its own past; cross-sectional
# demeaning compares assets with peers on one date; residualization removes the
# stated beta exposure in this controlled example.
n_assets, T_p = 8, 1500
demo3_rng = np.random.default_rng(0)
market = demo3_rng.standard_normal(T_p) * 0.01
idio = demo3_rng.standard_normal((T_p, n_assets)) * 0.01
asset_beta = np.linspace(0.4, 1.6, n_assets)
panel_r = market[:, None] * asset_beta[None, :] + idio
panel = pd.DataFrame(panel_r, columns=[f"a{i}" for i in range(n_assets)])

# At a post-close decision, today's return is known. Its own trailing reference
# ends yesterday so that today's observation does not mask itself.
panel_past = panel.shift(1)
ts_mean = panel_past.rolling(63).mean()
ts_std = panel_past.rolling(63).std()
panel_ts = (panel - ts_mean) / ts_std
panel_xs = panel.sub(panel.mean(axis=1), axis=0)

# Known-beta residualization is supplied as an exposure example. In real work,
# beta estimates and the eligible peer set require point-in-time definitions.
exposure_design = np.column_stack([np.ones(n_assets), asset_beta])
projection = exposure_design @ np.linalg.pinv(exposure_design)
panel_beta_neutral = pd.DataFrame(
    panel.to_numpy() @ (np.eye(n_assets) - projection),
    index=panel.index,
    columns=panel.columns,
)

row = 108
comparison = pd.DataFrame(
    {
        "raw return": panel.iloc[row],
        "time-series z vs own past": panel_ts.iloc[row],
        "cross-sectional demean": panel_xs.iloc[row],
        "known-beta residual": panel_beta_neutral.iloc[row],
    }
)
print(f"Comparison at simulated date {row}:")
print(comparison.round(4).to_string())
print(
    "\nCorrelation with stated beta at this date: "
    f"raw={np.corrcoef(panel.iloc[row], asset_beta)[0, 1]:+.3f}, "
    f"demeaned={np.corrcoef(panel_xs.iloc[row], asset_beta)[0, 1]:+.3f}, "
    f"residual={np.corrcoef(panel_beta_neutral.iloc[row], asset_beta)[0, 1]:+.3f}"
)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
panel.cumsum().plot(ax=axes[0], legend=False, lw=0.7)
axes[0].set_title("Raw cumulative returns")
panel_ts["a0"].plot(ax=axes[1], color="tab:blue", lw=0.7)
axes[1].axhline(0, color="0.2", lw=0.5)
axes[1].set_title("Asset a0 vs. its own past")
panel_xs.cumsum().plot(ax=axes[2], legend=False, lw=0.7)
axes[2].set_title("Returns vs. peers each date")
panel_beta_neutral.cumsum().plot(ax=axes[3], legend=False, lw=0.7)
axes[3].set_title("After known-beta adjustment")
fig.tight_layout(); plt.show()


# --- Scaler comparison on a fat-tailed feature (2A transform toolbox) ---
# One big jump inflates the SD, so StandardScaler compresses the bulk toward 0.
# RobustScaler divides by the IQR, which the jump does not move, so the bulk
# keeps its spread and the jump is left visibly far out.
ft = demo3_rng.standard_normal(400)
jump_idx = np.array([10, 20, 30, 350])
ft[jump_idx] += 30.0                               # a few fat-tailed jumps
ft = ft.reshape(-1, 1)
train_cut = 300
std_scaler = StandardScaler().fit(ft[:train_cut])
rob_scaler = RobustScaler().fit(ft[:train_cut])
std_scaled = std_scaler.transform(ft).ravel()
rob_scaled = rob_scaler.transform(ft).ravel()
bulk_train = np.setdiff1d(np.arange(train_cut), jump_idx[jump_idx < train_cut])
test_jump = 350
print("\nSame fat-tailed feature, two scalers fitted only through row 299:")
print(f"  StandardScaler: training-bulk SD={std_scaled[bulk_train].std():.2f}, "
      f"later shock={std_scaled[test_jump]:+.1f}")
print(f"  RobustScaler:   training-bulk SD={rob_scaled[bulk_train].std():.2f}, "
      f"later shock={rob_scaled[test_jump]:+.1f}")
print("  Fit cutoff is frozen; the later shock changes neither scaler.")
# Expected: demeaning leaves beta structure while residualization removes it;
#           both scalers use only the declared training population.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| comparison row and peers | `[row and eligible assets]` |
| asset `a7` under three comparisons | `[raw, own-history z, and peer-demeaned values]` |
| known-beta residual | `[residual value and what was removed]` |
| beta relationship across peers | `[raw, demeaned, and residual correlations]` |
| fitted scaler boundary | `[fit rows and freeze rule]` |
| later extreme under each ruler | `[StandardScaler and RobustScaler values]` |
| limitation | `[what the supplied comparison cannot establish]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed comparison entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| comparison row and peers | simulated row 108; assets `a0`–`a7` are the eligible peers |
| asset `a7` under three comparisons | raw **−0.0338**; own-history z-score **−1.98**; peer-demeaned **−0.0125** |
| known-beta residual | **−0.0035** after removing the supplied intercept and beta exposures |
| beta relationship across peers | correlation with beta is **−0.732** raw, **−0.732** after demeaning, and approximately **0** after residualization |
| fitted scaler boundary | both scalers are fitted only on rows 0–299 and then frozen |
| later extreme under each ruler | StandardScaler **+9.4**; RobustScaler **+19.9** |
| limitation | the beta is supplied by the simulation, and none of these transformations establishes later predictive value |

Demeaning changes the peer average but does not remove the beta relationship.
The later observation also stays fixed while the fitted ruler changes.

</details>

<a id="demo-4-define-and-align-a-forward-return-target"></a>

### Demo 4 — Define and align a forward-return target · Deck A · run at the break after S3

> **Break cue:** Deck A, after recording segment S3. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
ret, feat, row_ledger, timeline, df, audit_columns = build_week2_row_ledger()
assert (timeline["latest_source_time"] <= timeline["decision_time"]).all()
assert (timeline["decision_time"] < timeline["earliest_entry"]).all()
assert (timeline["target_start"] <= timeline["target_end"]).all()

print("One complete mature row:")
print(df[audit_columns].iloc[-1].to_string())
print("\nFinal row preserves the unavailable outcome:")
print(row_ledger[audit_columns].iloc[-1].to_string())
print(f"\nAligned dataset shape: {df.shape}")
print(f"First valid decision: {df['decision_time'].min()}")
print(f"Last valid decision:  {df['decision_time'].max()}")
print(f"Final raw target remains missing: {pd.isna(row_ledger['target_return'].iloc[-1])}")

# Sanity: features look back, target looks forward — they should not be
# mechanically correlated by construction.
for col in ["r_lag1_t", "mom5_t", "vol21_t"]:
    print(f"corr({col}, target_return) = {df[col].corr(df['target_return']):+.3f}")

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| decision time | `[timestamp]` |
| latest feature source | `[timestamp]` |
| feasible entry | `[timestamp]` |
| target interval and value | `[start, end, and return]` |
| aligned sample | `[first and last valid decision times]` |
| final unavailable label | `[value and maturity flag]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed target-ledger entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| decision time | 2022-08-31 16:05 |
| latest feature source | 2022-08-30 16:00 |
| feasible entry | 2022-09-01 09:30 |
| target interval and value | 2022-09-01 09:30 to 16:00; simulated open-to-close return **+0.008602** |
| aligned sample | first valid decision 2016-01-19 16:05; last valid decision 2022-08-31 16:05 |
| final unavailable label | `target_return = NaN`, `label_matured = False` |

The missing final target is correct evidence: the outcome had not happened and
must not be silently turned into zero or a class label.

</details>

<a id="demo-5-look-ahead-self-check-on-the-full-feature-matrix"></a>

### Demo 5 — Look-ahead self-check on the full feature matrix · Deck A · run at the break after S3

> **Break cue:** Deck A, after recording segment S3. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
ret, feat, row_ledger, timeline, df, audit_columns = build_week2_row_ledger()

def selfcheck_series(build_fn, x, cut, n_perms=4, seed=0):
    rng_local = np.random.default_rng(seed)
    base = build_fn(x)[:cut]
    max_change = 0.0
    for _ in range(n_perms):
        x2 = x.copy()
        idx = np.arange(cut, len(x))
        rng_local.shuffle(idx)
        x2[cut:] = x[idx]
        cand = build_fn(x2)[:cut]
        diff = np.nanmax(np.abs(np.asarray(base) - np.asarray(cand)))
        max_change = max(max_change, diff)
    return max_change


def build_past_only_feature_matrix(x):
    """Rebuild every return-derived feature using only information before t."""
    series = pd.Series(x)
    matrix = pd.DataFrame(index=series.index)
    for lag in [1, 2, 3, 5]:
        matrix[f"r_lag{lag}_t"] = series.shift(lag)
    for window in [5, 21, 63]:
        matrix[f"mom{window}_t"] = series.rolling(window).mean().shift(1)
    for window in [21, 63]:
        matrix[f"vol{window}_t"] = series.rolling(window).std().shift(1)
    mom21_known = series.shift(1).rolling(21).mean()
    matrix["zmom21_t"] = (
        (mom21_known - mom21_known.rolling(252).mean())
        / mom21_known.rolling(252).std()
    )
    matrix["dow_t"] = np.arange(len(series)) % 5
    return matrix.to_numpy()


def build_mom21_leaky(x):
    return pd.Series(x).rolling(21, center=True).mean().values

x_arr = ret.values
cut = int(0.7 * len(x_arr))
print(
    "Full past-only feature matrix max change: "
    f"{selfcheck_series(build_past_only_feature_matrix, x_arr, cut):.2e}"
)
print(f"Leaky   mom21 (centered)  max change:    {selfcheck_series(build_mom21_leaky, x_arr, cut):.2e}")


def assert_row_timing(frame):
    complete = frame.dropna(
        subset=["latest_source_time", "decision_time", "target_start", "target_end"]
    )
    assert (complete["latest_source_time"] <= complete["decision_time"]).all()
    assert (complete["decision_time"] < complete["target_start"]).all()
    assert (complete["target_start"] <= complete["target_end"]).all()


assert_row_timing(row_ledger)
bad_ledger = row_ledger.copy()
bad_row = timeline.index[0]
bad_ledger.loc[bad_row, "latest_source_time"] = (
    bad_ledger.loc[bad_row, "decision_time"] + pd.Timedelta(minutes=1)
)
try:
    assert_row_timing(bad_ledger)
except AssertionError:
    print(f"Expected failure: row {bad_row} touches a source after its decision time.")

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| CLAIM | EVIDENCE | BOUNDARY | LIMITATION |
|:--|:--|:--|:--|
| `[what stayed unchanged and what failed]` | `[both maximum changes plus the intentional assertion failure]` | `[cutoff, permutations, and clocks held fixed]` | `[what these tests cannot certify]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed invariance entry**

| CLAIM | EVIDENCE | BOUNDARY | LIMITATION |
|:--|:--|:--|:--|
| The full past-only feature matrix is future-invariant; a centered `mom21` is not. | Maximum pre-cutoff change: full matrix **0.00**; centered momentum **2.27×10⁻³**. The row-timing assertion also rejects a source timestamp one minute after its decision. | One simulated feature matrix, cutoff at row 1400, and four future permutations; target and feature clocks are held fixed. | Passing cannot certify vendor timestamps, dated universe membership, or whether the declared economic decision time is sensible. |

Record both what the test proves and what it cannot see.

</details>

<a id="demo-6-manufacture-a-leak-then-measure-it"></a>

### Demo 6 — Manufacture a leak, then measure it · Deck B · run at the break after S4

> **Break cue:** Deck B, after recording segment S4. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
# Known truth: returns are i.i.d., so NO feature built from the past can predict
# them. The correct out-of-sample R-squared is zero. Anything above zero here was
# manufactured by the construction, not discovered in the market.
T = 4000
demo6_rng = np.random.default_rng(6)
ret = pd.Series(demo6_rng.standard_normal(T) * 0.01, name="ret")

# The target: tomorrow's return, as a decision at the close of day t would face.
target = ret.shift(-1)

# Two features, identical in every respect except the window's alignment.
leaky = ret.rolling(21, center=True).mean()        # window straddles day t
clean = ret.rolling(21).mean().shift(1)            # window ends before day t

split = int(T * 0.7)


def forward_r2(feature, label):
    """Fit on the first 70% of dates, score on the untouched later block."""
    frame = pd.concat([feature.rename("x"), label.rename("y")], axis=1).dropna()
    train, test = frame.iloc[:split], frame.iloc[split:]
    model = LinearRegression().fit(train[["x"]], train["y"])
    pred = model.predict(test[["x"]])
    baseline = train["y"].mean()          # the feasible zero-edge forecast
    ss_res = ((test["y"] - pred) ** 2).sum()
    ss_bas = ((test["y"] - baseline) ** 2).sum()
    return 1 - ss_res / ss_bas


print("Known truth: returns are i.i.d.; the honest forward R-squared is 0.")
print(f"  centered window (leaks day t+10):  forward R2 = {forward_r2(leaky, target):+.4f}")
print(f"  trailing window, shifted:          forward R2 = {forward_r2(clean, target):+.4f}")
print()

# How far into the future does the leak reach, and how much edge does each day buy?
print("Edge manufactured as the window is slid off the future:")
for lead in range(10, -1, -2):
    feat = ret.rolling(21).mean().shift(-lead)
    label = "  (no future)" if lead == 0 else f"  (sees {lead} day(s) ahead)"
    print(f"  shift={-lead:+d}: forward R2 = {forward_r2(feat, target):+.4f}{label}")

print()
print("Nothing was learned about markets. The only thing that changed is whether")
print("the feature was allowed to see the day it was asked to predict.")
# Expected: truth is 0. The centered window earns roughly +0.04 to +0.06, the trailing
#           window ~0.000, and the decay table falls toward 0 as look-ahead is removed.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| the true R² | `[known value and why]` |
| leaked value | `[score and feature construction]` |
| clean value | `[score and repaired construction]` |
| days of look-ahead | `[how far the leaked window reaches]` |
| the repair | `[change that restores the boundary]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**What a complete entry looks like** — the slide asks for *the true R²; the
leaked and clean values; how many days of look-ahead the feature had; the repair
that removed it*.

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| the true R² | **exactly 0.** Returns are generated i.i.d., so no feature built from the past can forecast them |
| leaked value | forward R² = **+0.0567** using a centered 21-day window |
| clean value | forward R² = approximately **0.0000** using the same window, trailing and shifted |
| days of look-ahead | 10 — a centered window at day *t* reaches to *t*+10 |
| the repair | make the window trailing and shift it by one, so the last input is *t*−1 |

Read the decay table underneath: +0.0567, +0.0579, +0.0548, +0.0509, +0.0446,
and approximately 0.0000 at 10, 8, 6, 4, 2, and 0 future days. It does not fall smoothly,
and that is worth noticing—the size of a leak is not a reliable signal of how
much future the feature saw. Only the construction tells you that.

Every one of those positive numbers is manufactured. Nothing was learned about
markets.

</details>

<a id="demo-7-full-sample-normalization-is-an-invisible-leak"></a>

### Demo 7 — Full-sample normalization is an invisible leak · Deck B · run at the break after S5

> **Break cue:** Deck B, after recording segment S5. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
# Series that is calm then turbulent. Two z-scores: full-sample vs. rolling.
T = 1500
demo7_rng = np.random.default_rng(2)
calm = demo7_rng.standard_normal(T // 2) * 0.005
turb = demo7_rng.standard_normal(T // 2) * 0.025
x = np.concatenate([calm, turb])
xp = pd.Series(x)

cut = T // 2
z_full = (xp - xp.mean()) / xp.std()  # LEAKY
train_mean = xp.iloc[:cut].mean()
train_std = xp.iloc[:cut].std()
z_train_fit = (xp - train_mean) / train_std

fig, axes = plt.subplots(3, 1, figsize=(10, 5.5), sharex=True)
axes[0].plot(xp, lw=0.5); axes[0].set_title("Raw series (calm → turbulent)")
axes[1].plot(z_full, color="tab:orange", lw=0.5)
axes[1].set_title("Full-sample z-score (LEAKY — uses future stats)")
axes[2].plot(z_train_fit, color="tab:blue", lw=0.5)
axes[2].set_title("Training-fit z-score (frozen before the test period)")
for ax in axes:
    ax.axvline(cut, color="0.2", ls="--", lw=1)
fig.tight_layout(); plt.show()

print(f"Full-sample scale: {xp.std():.4f}")
print(f"Training-only scale: {train_std:.4f}")
print(
    "First test observation | full-sample z = "
    f"{z_full.iloc[cut]:+.3f}, training-fit z = {z_train_fit.iloc[cut]:+.3f}"
)
# Expected: the full-sample scale (~0.018) is several times the training-only scale
#           (~0.005), moving the first test z from about -0.5 to -2.0.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| fitted state and cutoff | `[quantity learned and final training row]` |
| full-sample scale | `[value and population used]` |
| training-only scale | `[value and population used]` |
| first test observation | `[both transformed values]` |
| why they differ | `[information-path explanation]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**What a complete entry looks like** — the slide asks for *the fitted state and
its training cutoff; both scale estimates; why the corrected test values
differ*.

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| fitted state and cutoff | one scale parameter (standard deviation), fitted on rows 0–749 only |
| full-sample scale | **0.0184** — computed over calm *and* turbulent periods together |
| training-only scale | **0.0050** — the calm period alone, which is all a forecaster had |
| first test observation | full-sample z = **−0.509**; training-fit z = **−2.026** |
| why they differ | the full-sample ruler already knows the turbulence is coming, so it makes the early move look ordinary |

The raw return is identical in both cases. Only the ruler changed. A −2.0 that
gets reported as −0.5 is the difference between an alarming day and an
unremarkable one, and no amount of downstream modelling recovers it.

</details>

<a id="demo-8-point-in-time-alignment-with-an-as-of-join"></a>

### Demo 8 — Point-in-time alignment with an as-of join · Deck B · run at the break after S6

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
decisions = pd.DataFrame(
    {
        "entity": ["A", "C", "A", "C", "A", "C", "A", "C"],
        "decision_time": pd.to_datetime(
            [
                "2024-05-01 16:05", "2024-05-01 16:05",
                "2024-05-06 16:05", "2024-05-06 16:05",
                "2024-08-01 16:05", "2024-08-01 16:05",
                "2024-08-08 16:05", "2024-08-08 16:05",
            ]
        ),
    }
).sort_values(["decision_time", "entity"])
vintages = pd.DataFrame(
    {
        "entity": ["A", "C", "A", "C", "A", "A"],
        "period_end": pd.to_datetime(
            [
                "2023-12-31", "2023-12-31", "2024-03-31",
                "2024-03-31", "2024-03-31", "2024-03-31",
            ]
        ),
        "available_time": pd.to_datetime(
            [
                "2024-02-15", "2024-02-20", "2024-05-03",
                "2024-05-10", "2024-08-06", "2025-01-15",
            ]
        ),
        "vintage": [
            "2023Q4 initial", "2023Q4 initial", "2024Q1 initial",
            "2024Q1 initial", "2024Q1 revised", "2024Q1 benchmark",
        ],
        "revenue": [92.0, 41.0, 100.0, 44.0, 96.0, 97.0],
    }
).sort_values(["available_time", "entity"])

pit = pd.merge_asof(
    decisions,
    vintages,
    by="entity",
    left_on="decision_time",
    right_on="available_time",
    direction="backward",
)
assert (pit["available_time"] <= pit["decision_time"]).all()
print(pit.to_string(index=False))

# The same economic period can have different values under different
# information dates. The historical backtest needs the left-hand value; a
# database downloaded later may silently supply the right-hand one.
historical_row = pit[
    (pit["entity"] == "A")
    & (pit["decision_time"] == pd.Timestamp("2024-08-01 16:05"))
].iloc[0]
benchmark_row = vintages[
    (vintages["entity"] == "A")
    & (vintages["period_end"] == pd.Timestamp("2024-03-31"))
].sort_values("available_time").iloc[-1]
print("\nOne target, two information dates:")
print(
    "  X_(2024Q1 | 2024-08-01) = "
    f"{historical_row['revenue']:.0f} ({historical_row['vintage']})"
)
print(
    "  X_(2024Q1 | 2025-01-15) = "
    f"{benchmark_row['revenue']:.0f} ({benchmark_row['vintage']})"
)
print("The second value may be a better estimate, but it was not knowable at the first date.")

# Now repeat the idea with actual ALFRED real-GDP vintages. The compact extract
# is bundled with the lecture and embedded here as a fallback so this cell also
# runs in a standalone Colab notebook without a network connection or API key.
gdp_extract_csv = """economic_date,first_print,plus_3_months,plus_1_year,current_benchmark
2008-10-01,-0.9647894539,-1.6247737441,-1.3711481020,-2.1890265953
2009-01-01,-1.5726299893,-1.4025221097,-1.6474025826,-1.1348742975
2014-01-01,0.0269722687,-0.5309056182,-0.5309056182,-0.3450959853
2014-04-01,0.9727319239,1.1287480182,1.1230596061,1.2918421292
"""
gdp_vintages, gdp_source = bundled_csv(
    "lectures/week02/data/alfred_gdpc1_vintages.csv",
    gdp_extract_csv,
)
gdp_vintages["economic_date"] = pd.to_datetime(gdp_vintages["economic_date"])
gdp_matrix = gdp_vintages.set_index("economic_date")
gdp_matrix.index = gdp_matrix.index.to_period("Q").astype(str)
gdp_matrix.columns = ["first print", "+3 months", "+1 year", "current benchmark"]

print("\nActual ALFRED real-GDP vintage matrix (quarter-over-quarter growth, %):")
print(gdp_matrix.round(2).to_string())
print(f"Source used by this run: {gdp_source}")

fig, ax = plt.subplots(figsize=(9, 3.3))
values = gdp_matrix.to_numpy()
ax.imshow(values, cmap="RdBu", vmin=-2.3, vmax=2.3, aspect="auto")
ax.set_xticks(range(len(gdp_matrix.columns)), gdp_matrix.columns)
ax.set_yticks(range(len(gdp_matrix.index)), gdp_matrix.index)
ax.xaxis.tick_top()
for row in range(values.shape[0]):
    for col in range(values.shape[1]):
        color = "white" if abs(values[row, col]) > 1.0 else "#17365d"
        ax.text(col, row, f"{values[row, col]:+.2f}%", ha="center", va="center",
                color=color, fontweight="bold")
for col in [0, 1, 3]:
    ax.add_patch(plt.Rectangle((col - 0.48, 2 - 0.46), 0.96, 0.92,
                               fill=False, edgecolor="#b07d20", linewidth=2.5))
ax.set_title("The economic date stays fixed while the published history changes", pad=34)
for spine in ax.spines.values():
    spine.set_visible(False)
fig.tight_layout(); plt.show()

gdp_2014q1 = gdp_matrix.loc["2014Q1"]
first_sign = "expansion" if gdp_2014q1["first print"] > 0 else "contraction"
later_sign = "expansion" if gdp_2014q1["current benchmark"] > 0 else "contraction"
print("\n2014 Q1 sign-based feature:")
print(f"  first print:       {gdp_2014q1['first print']:+.2f}% → {first_sign}")
print(f"  three months on:   {gdp_2014q1['+3 months']:+.2f}% → contraction")
print(f"  current benchmark: {gdp_2014q1['current benchmark']:+.2f}% → {later_sign}")
assert first_sign == "expansion" and later_sign == "contraction"
print("A forward split would not detect the replacement of the first-print feature with the current one.")

# A point-in-time universe check uses dated membership, not today's survivor list.
membership = pd.DataFrame(
    {
        "entity": ["A", "B", "C"],
        "eligible_from": pd.to_datetime(["2024-01-01", "2024-03-01", "2024-01-01"]),
        "eligible_to": pd.to_datetime([None, "2024-07-15", "2024-09-01"]),
        "exit_outcome": [None, "delisted", "acquired"],
    }
)
universe_date = pd.Timestamp("2024-08-01")
eligible = membership[
    (membership["eligible_from"] <= universe_date)
    & (membership["eligible_to"].isna() | (universe_date < membership["eligible_to"]))
]
assert "B" not in eligible["entity"].tolist()
print(f"\nEligible universe on {universe_date.date()}:")
print(eligible.to_string(index=False))
print("Entity B is absent because its dated membership ended before the decision.")

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| grouped join rule | `[direction, entity key, and time key]` |
| entity A on 2024-08-01 | `[vintage, value, and availability]` |
| entity A on 2024-08-08 | `[vintage, value, and availability]` |
| entity C on 2024-08-08 | `[value showing that entity histories stayed separate]` |
| two-date comparison | `[$X_{t\mid s}$ at the historical decision and $X_{t\mid T}$ at the later benchmark]` |
| real GDP vintage row | `[2014 Q1 first-print, three-month, and current values; resulting sign feature]` |
| source and construction | `[series, download date, transformation, and meaning of current]` |
| dated universe | `[eligible and excluded entities, with the rule]` |
| limitation | `[what this extract and correct join mechanics cannot certify]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed as-of entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| grouped join rule | backward as-of join by `entity`, choosing the latest `available_time` no later than the decision |
| entity A on 2024-08-01 | 2024Q1 initial revenue **100**, available 2024-05-03 |
| entity A on 2024-08-08 | 2024Q1 revised revenue **96**, available 2024-08-06 |
| entity C on 2024-08-08 | its own 2024Q1 initial revenue **44**; A's revision cannot cross the entity boundary |
| two-date comparison | $X_{2024Q1\mid 2024-08-01}=100$ but the later benchmark is $X_{2024Q1\mid 2025-01-15}=97$; the latter was not historically admissible |
| real GDP vintage row | 2014 Q1 changes from **+0.03%** at first print to **−0.53%** three months later and **−0.35%** in the current benchmark; the sign-based feature changes from expansion to contraction |
| source and construction | ALFRED `GDPC1`, downloaded 2026-08-29; quarter-over-quarter growth computed within each vintage; current means latest available benchmark, not final truth |
| dated universe | A and C are eligible on 2024-08-01; B is absent because its membership ended 2024-07-15 |
| limitation | the four-row extract demonstrates revision mechanics but does not measure predictive-skill retention; correct mechanics also cannot prove that vendor timestamps or membership history are complete |

The current database row is not automatically the row that existed at an
earlier decision date.

</details>

<a id="demo-9-the-look-ahead-self-check"></a>

### Demo 9 — The look-ahead self-check · Deck B · run at the break after S6

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
def lookahead_selfcheck(feature_fn, x, cut, n_perms=5, seed=0):
    """Scramble x after `cut`; the feature's pre-cut values must not change.
    Returns the max absolute change observed across permutations."""
    rng_local = np.random.default_rng(seed)
    base = feature_fn(x)[:cut]
    max_change = 0.0
    for _ in range(n_perms):
        x_scrambled = x.copy()
        tail_idx = np.arange(cut, len(x))
        rng_local.shuffle(tail_idx)
        x_scrambled[cut:] = x[tail_idx]
        cand = feature_fn(x_scrambled)[:cut]
        diff = np.nanmax(np.abs(base - cand))
        max_change = max(max_change, diff)
    return max_change

demo9_rng = np.random.default_rng(9)
x = demo9_rng.standard_normal(1000)

# CORRECT: rolling mean, then shift(1) — touches data through t-1 only
def feat_correct(s):
    return pd.Series(s).rolling(20).mean().shift(1).values

# LEAKY: centered rolling window — averages over data at and after t
def feat_leaky(s):
    return pd.Series(s).rolling(20, center=True).mean().values

cut = 700
chg_correct = lookahead_selfcheck(feat_correct, x, cut)
chg_leaky   = lookahead_selfcheck(feat_leaky,   x, cut)
print(f"Past-only feature  max change of past values: {chg_correct:.3e}  (should be 0)")
print(f"Centered feature   max change of past values: {chg_leaky:.3e}  (LEAK — past sees future)")
# Expected: the past-only feature changes past values by exactly 0; the centered one by
#           ~0.36. Any nonzero number here is a leak, not a rounding artifact.

#### Your completed seven-field information-boundary record

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| `decision_at` | `[when the model acts]` |
| `feature_latest_at` | `[newest source observation admitted]` |
| `feature_window` | `[earliest and latest inputs used]` |
| `label_interval` | `[start and end of the outcome]` |
| `source_vintage` | `[economic date, availability, entity, and revision identity]` |
| `universe_asof` | `[dated membership rule]` |
| `fit_cutoff` | `[latest row used to fit any transformation, or not applicable]` |

**Verification evidence**

| CHECK | YOUR ENTRY |
|:--|:--|
| as-of assertion | `[availability-time result and entity grouping]` |
| past-only future invariance | `[maximum pre-cutoff change]` |
| deliberately leaky comparison | `[maximum pre-cutoff change]` |
| route not ruled out | `[one important failure these checks cannot detect]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed seven-field information-boundary record**

This is one illustrative entity-A row assembled from the clock and point-in-time
rules used this week.

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| `decision_at` | 2024-08-01 16:05 |
| `feature_latest_at` | 2024-05-03, when the selected revenue vintage became available |
| `feature_window` | one 2024Q1 accounting observation with economic period end 2024-03-31 |
| `label_interval` | next-session open to close: 2024-08-02 09:30–16:00 |
| `source_vintage` | entity A, 2024Q1 initial revenue 100; available 2024-05-03; later revision not yet admitted |
| `universe_asof` | entity A is eligible on 2024-08-01 under its dated membership interval |
| `fit_cutoff` | not applicable to this raw as-of feature; any fitted transform must add its own cutoff |

**Verification evidence from Demos 8–9**

| CHECK | RESULT |
|:--|:--|
| as-of assertion | every selected `available_time` is no later than `decision_time`, with joins grouped by entity |
| past-only future invariance | maximum pre-cutoff change **0.000** |
| deliberately leaky comparison | centered feature maximum pre-cutoff change **0.5558** |
| route not ruled out | a historically wrong source timestamp, omitted revision, or incomplete membership history |

Changing tomorrow should not move yesterday. Passing that test is necessary,
not sufficient, evidence of a point-in-time pipeline.

</details>

<a id="demo-10-a-spectacular-backtest-built-entirely-from-a-leak"></a>

### Demo 10 — A spectacular backtest built entirely from a leak · Deck B · run at the break after S6

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one claim the result cannot establish.

In [ ]:
#
# Known truth: returns are i.i.d. noise. No strategy can have an edge, so the
# honest Sharpe ratio is zero. The only difference between the two strategies
# below is whether the signal's window is allowed to touch the day it trades.
T = 3000
demo10_rng = np.random.default_rng(20)
ret10 = pd.Series(demo10_rng.standard_normal(T) * 0.01, name="ret")


def sharpe(series):
    """Annualized Sharpe of a daily return series."""
    return float(series.mean() / series.std() * np.sqrt(252))


# The leak: a centered window straddles day t, so the signal already contains
# part of the return it is about to trade. Nobody writes this on purpose; it is
# what `center=True` does, and it looks like an ordinary moving average.
leaked_signal = ret10.rolling(5, center=True).mean()
leaked_pnl = (np.sign(leaked_signal) * ret10).dropna()

# The repair: the window ends before the decision, and the position is held into
# the following day's return. Same data, same rule, same code path.
honest_signal = ret10.rolling(5).mean().shift(1)
honest_pnl = (np.sign(honest_signal) * ret10).dropna()

print("Known truth: returns are i.i.d., so the honest Sharpe is 0.")
print(f"  leaked (centered window):  Sharpe = {sharpe(leaked_pnl):.2f}")
print(f"  honest (trailing, shifted): Sharpe = {sharpe(honest_pnl):.2f}")
print()
print(f"  leaked cumulative return:  {leaked_pnl.sum():+.2%}")
print(f"  honest cumulative return:  {honest_pnl.sum():+.2%}")
print()
print("A Sharpe above 6 on a series with no signal at all. The backtest is not")
print("evidence of skill; it is evidence that one window was aligned wrongly.")

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(leaked_pnl.cumsum().values, color="#c62828", lw=1.6,
        label=f"leaked window — Sharpe {sharpe(leaked_pnl):.2f}")
ax.plot(honest_pnl.cumsum().values, color="#2e7d32", lw=1.6,
        label=f"same rule, window shifted — Sharpe {sharpe(honest_pnl):.2f}")
ax.axhline(0, color="0.4", lw=0.8)
ax.set_xlabel("trading day"); ax.set_ylabel("cumulative return")
ax.set_title("Both lines trade the same noise. Only one saw the day it traded.")
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); plt.show()